# Nguyễn Thành Trí - MSSV: 6451071079
## Phần 2 - Bài 5a (Trang 45) - Câu 8: Xóa cột 'POSTAGE' khỏi ma trận One-Hot Encoding

### Đề bài:
Do cột `'POSTAGE'` là tiền cước phí bưu điện/vận chuyển trên mỗi hóa đơn nên cần xóa nó đi khỏi ma trận phân tích giỏ hàng.

In [1]:
# Nguyễn Thành Trí - 6451071079
# Phần 2 - Bài 5a (Trang 45) - Câu 8: Xóa cột 'POSTAGE'

import os
import pandas as pd
import numpy as np

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def load_clean_data():
    pkl_candidates = ["datasets/online_retail_clean.pkl", "../datasets/online_retail_clean.pkl", "../../datasets/online_retail_clean.pkl"]
    for p in pkl_candidates:
        if os.path.exists(p):
            return pd.read_pickle(p)
    file_path = "datasets/Online Retail.xlsx" if os.path.exists("datasets/Online Retail.xlsx") else "../datasets/Online Retail.xlsx"
    df = pd.read_excel(file_path)
    df['Description'] = df['Description'].str.strip()
    df = df.dropna(subset=['InvoiceNo'])
    df['InvoiceNo'] = df['InvoiceNo'].astype(str)
    return df[~df['InvoiceNo'].str.contains('C', na=False)]

df = load_clean_data()
df_germany = df[df['Country'] == 'Germany']

basket = (df_germany.groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

basket_sets = (basket >= 1)
print(f"Số lượng cột trước khi xử lý POSTAGE: {basket_sets.shape[1]}")

# Kiểm tra sự hiện diện của cột POSTAGE trong dữ liệu Đức
if 'POSTAGE' in basket_sets.columns:
    postage_count = basket_sets['POSTAGE'].sum()
    postage_rate = postage_count / len(basket_sets) * 100
    print(f"Cột 'POSTAGE' xuất hiện trong {postage_count}/{len(basket_sets)} hóa đơn (chiếm {postage_rate:.2f}% tổng đơn hàng Đức).")
    basket_sets = basket_sets.drop(columns=['POSTAGE'])
    print("Đã xóa thành công cột 'POSTAGE'.")
else:
    print("Không tìm thấy cột 'POSTAGE'.")

print(f"Số lượng cột sau khi xử lý: {basket_sets.shape[1]}")

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
Số lượng cột trước khi xử lý POSTAGE: 1695
Cột 'POSTAGE' xuất hiện trong 374/457 hóa đơn (chiếm 81.84% tổng đơn hàng Đức).
Đã xóa thành công cột 'POSTAGE'.
Số lượng cột sau khi xử lý: 1694


### Nhận xét & Đánh giá (Câu 8):
1. **Tầm quan trọng đặc biệt đối với thị trường Đức**: Cột `'POSTAGE'` xuất hiện trong **374 trên tổng số 457** hóa đơn tại Đức (chiếm tỷ lệ cực lớn: **81.84%**). Do đây là đơn hàng quốc tế từ Anh gửi sang Đức, gần như mọi đơn hàng đều phát sinh chi phí bưu chính quốc tế.
2. **Lý do bắt buộc phải loại bỏ**: Nếu không xóa `'POSTAGE'`, thuộc tính này sẽ có `support` lên tới 81.84% và xuất hiện trong hầu như tất cả các luật kết hợp (ví dụ: "Khách mua Hộp đồ ăn nhẹ -> Trả cước bưu điện"). Đây là các luật hiển nhiên, vô nghĩa về mặt kinh doanh và làm loãng các tri thức giá trị về mối liên kết giữa các sản phẩm quà tặng thực sự.